# B. Operation: EchoTrace fixed — поиск похожих изображений

Для каждого из 9605 изображений требуется найти шесть других похожих кадров. Извлекаем 2048 признаков предобученной ResNet-50 (ImageNet V2), нормализуем векторы и выбираем шесть ближайших по косинусному сходству. Обучение не требуется.

Предобработка соответствует весам: RGB, resize 232, центральная обрезка 224×224, нормализация ImageNet. Последний классифицирующий слой заменён на Identity. При поиске соседей сходство изображения с собой устанавливается в минус бесконечность.

Признаки сохраняются в `features.npz`; при наличии кэша повторный запуск выполняет только поиск соседей. Для расчёта используется GPU MPS, если доступен, иначе CPU.

Документация: https://docs.pytorch.org/vision/0.16/models/generated/torchvision.models.resnet50.html

## Запуск

Изображения из архива должны лежать в `task_b/data`, веса — в `task_b/weights`. Требуются torch, torchvision, numpy и Pillow. Скрипт и README содержат детали воспроизведения.


In [1]:
from pathlib import Path
import runpy
folder = Path.cwd() if Path('retrieve.py').exists() else Path.cwd() / 'task_b'
runpy.run_path(str(folder / 'retrieve.py'), run_name='__main__');

Device: cpu
{
  "images": 9605,
  "recommendations_per_image": 6,
  "model": "torchvision ResNet50 IMAGENET1K_V2",
  "features": 2048,
  "metric": "cosine similarity",
  "format_checks": "passed",
  "contest_score": null
}


## Проверки и ограничения

Скрипт проверяет, что каждый входной файл представлен ровно одной строкой CSV, все шесть соседей существуют, не повторяются и не совпадают с запросом. Итоги сохраняются в `validation.json`.

`submission.csv` загружается в задачу B. Файл `neighbours.jpg` предназначен для визуальной проверки: слева запрос, справа шесть соседей.

Косинусное сходство нашей модели не равно метрике контеста. Эталонные рекомендации отсутствуют, поэтому достижение порога 0.5 локально не подтверждено. После отправки нужно добавить полученный результат контеста.
